# Fundamentals of Machine Learning — Exercise 2
## Exploratory Data Analysis and Statistical Properties

In this exercise, you will continue working with the **House Prices** dataset. The focus is not on writing long programs. Most code is already prepared. Your task is to use the output to answer analytical questions.

We will use the same workflow as in Exercise 1:

> **question → prediction → computation → visualisation → interpretation → verification**

### Learning outcomes

By the end of the exercise, you should be able to:

- distinguish a variable's analytical role from its Pandas `dtype`,
- choose between the mean, median, standard deviation, and IQR,
- describe the shape and spread of a numerical distribution,
- interpret an ordinal variable without treating it as a continuous measurement,
- investigate whether a missing value means “unknown” or “not applicable”,
- choose a graph based on an analytical question,
- interpret Pearson and Spearman correlation cautiously,
- flag unusual observations and investigate them before removing them,
- use an AI chatbot as a tutor without letting it replace your own explanation.

**Recommended duration:** 90 minutes


## How to work in this exercise

Most code is already available. Your task is to:

1. **predict** what you expect before running a cell,
2. **choose** a suitable statistic or visualisation,
3. **interpret** the result in your own words,
4. **verify** an important claim using another output,
5. **modify** only a small part of the code when the question changes.

Do not copy a chatbot's explanation into the notebook. When AI is used, you will first write your own answer, let the chatbot ask you questions, close the chat, and then explain the idea again in your own words.

### Suggested pair-work routine

- Student A predicts or explains the expected result.
- Student B runs the code and checks the output.
- Both agree on a short interpretation.
- Swap roles for the next activity.


## 0. Imports and dataset

We will use the same dataset as in Exercise 1. Each row represents one recorded house sale.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")


**Source:** [House Prices: Advanced Regression Techniques](https://www.kaggle.com/c/house-prices-advanced-regression-techniques/data)

The cell first looks for a local file named `zsu_cv1_data.csv`. If it is not available, it loads the course copy from GitHub.


In [ ]:
DATA_URL = (
    "https://raw.githubusercontent.com/rasvob/"
    "VSB-FEI-Fundamentals-of-Machine-Learning-Exercises/"
    "master/datasets/zsu_cv1_data.csv"
)

LOCAL_DATA_PATH = Path("zsu_cv1_data.csv")
data_source = LOCAL_DATA_PATH if LOCAL_DATA_PATH.exists() else DATA_URL

df_full = pd.read_csv(data_source)

print(f"Source: {data_source}")
print(f"Shape: {df_full.shape[0]} rows × {df_full.shape[1]} columns")
df_full.head()


## 1. Build an analytical map

Pandas reports how a column is stored. An analyst must also decide what the values **mean**.

We will focus on a smaller set of variables.


In [ ]:
focus_columns = [
    "Id",
    "MSSubClass",
    "OverallQual",
    "OverallCond",
    "GarageFinish",
    "BldgType",
    "Neighborhood",
    "YearBuilt",
    "GrLivArea",
    "GarageArea",
    "SalePrice",
]

df = df_full.loc[:, focus_columns].copy()

profile = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "unique_values": df.nunique(dropna=True),
    "example": [
        df[column].dropna().iloc[0] if df[column].notna().any() else np.nan
        for column in df.columns
    ],
})

profile


### Activity 1 — Assign analytical roles

For each variable, choose the most suitable role:

- **identifier**
- **categorical code**
- **ordinal variable**
- **continuous measurement**
- **nominal category**
- **target variable**

| Variable | Your role | Why? |
|---|---|---|
| `Id` |  |  |
| `MSSubClass` |  |  |
| `OverallQual` |  |  |
| `Neighborhood` |  |  |
| `GrLivArea` |  |  |
| `SalePrice` |  |  |

Then answer:

1. Why is `dtype` alone insufficient?
2. What incorrect assumption would we make if `MSSubClass` were treated as an ordinary measurement?


## 2. What is a typical sale price?

A numerical distribution can be summarised in several ways. The choice should depend on its shape and on the analytical question.

### Answer

What is mean, median and modus?

### Predict before running

1. Do you expect the mean sale price to be higher or lower than the median?
2. If one house is much more expensive than all the others, which value will change more: the mean price or the median price? Explain why.


In [ ]:
sale_price = df["SalePrice"].dropna()

q1 = sale_price.quantile(0.25)
median = sale_price.median()
q3 = sale_price.quantile(0.75)

sale_price_summary = pd.Series({
    "count": sale_price.count(),
    "mean": sale_price.mean(),
    "median": median,
    "standard_deviation": sale_price.std(),
    "Q1": q1,
    "Q3": q3,
    "IQR": q3 - q1,
    "minimum": sale_price.min(),
    "maximum": sale_price.max(),
    "skewness": sale_price.skew(),
})

sale_price_summary


## READING: Summary Statistics for House Prices

The `sale_price` variable contains the sale prices of houses. The following values provide a basic description of their distribution.

### Number of Values (`count`)

The number of non-missing values in the data column:

$$
n = \text{number of known sale prices}
$$

The `count()` method does not include missing values (`NaN`).

---

### Arithmetic Mean (`mean`)

The arithmetic mean is calculated by adding all prices and dividing the result by the number of prices:

$$
\bar{x} = \frac{1}{n}\sum_{i=1}^{n}x_i
$$

where:

- $x_i$ is the price of the $i$-th house,
- $n$ is the number of known prices,
- $\bar{x}$ is the arithmetic mean.

The mean is sensitive to extremely high or low prices.

---

### Median (`median`)

The median is the middle value after sorting the prices from the lowest to the highest.

- If the number of values is odd, the median is the middle price.
- If the number of values is even, the median is the mean of the two middle prices.

For example, for the sorted prices

$$
2,\ 3,\ 4,\ 8,\ 20
$$

the median is $4$.

The median is usually not strongly affected by a single extremely high or low price.

---

### Standard Deviation (`standard_deviation`)

The standard deviation describes how much the prices tend to differ from the mean:

$$
s = \sqrt{\frac{1}{n-1}\sum_{i=1}^{n}(x_i-\bar{x})^2}
$$

where:

- $x_i$ is the price of the $i$-th house,
- $\bar{x}$ is the mean price,
- $n$ is the number of prices,
- $s$ is the sample standard deviation.

Interpretation:

- A small standard deviation means that most prices are relatively close to the mean.
- A large standard deviation means that the prices differ considerably.

The `pandas.Series.std()` method calculates the sample standard deviation. Therefore, the formula divides by $n-1$.

---

### First Quartile (`Q1`)

The first quartile is the value below which approximately 25% of the prices lie:

$$
Q_1 = \text{25th percentile}
$$

Approximately one quarter of the houses therefore have a price less than or equal to $Q_1$.

---

### Third Quartile (`Q3`)

The third quartile is the value below which approximately 75% of the prices lie:

$$
Q_3 = \text{75th percentile}
$$

Approximately one quarter of the houses have a price greater than $Q_3$.

---

### Interquartile Range (`IQR`)

The interquartile range is the difference between the third and first quartiles:

$$
IQR = Q_3-Q_1
$$

It describes the range containing the middle 50% of the prices. Unlike the full range, it is not strongly affected by a small number of extremely high or low values.

The IQR can also be used to identify potential outliers. Values outside the following interval are often considered potential outliers:

$$
\left[Q_1-1.5\cdot IQR,\ Q_3+1.5\cdot IQR\right]
$$

This rule identifies unusual values, but it does not prove that they are incorrect.

---

### Minimum (`minimum`)

The lowest known sale price:

$$
x_{\min} = \min(x_1,x_2,\ldots,x_n)
$$

The minimum may indicate an unusually inexpensive house or a possible error in the data.

---

### Maximum (`maximum`)

The highest known sale price:

$$
x_{\max} = \max(x_1,x_2,\ldots,x_n)
$$

The maximum may indicate an unusually expensive house or a possible error in the data.

---

### Skewness (`skewness`)

Skewness describes whether the values are distributed approximately evenly around the centre or whether the distribution extends further in one direction.

A basic interpretation is:

- `skewness ≈ 0`: the distribution is approximately symmetrical,
- `skewness > 0`: the distribution extends further towards high values,
- `skewness < 0`: the distribution extends further towards low values.

House prices often have positive skewness because a small number of very expensive houses create a long right-hand tail.

The basic principle of skewness can be expressed using the third powers of the differences from the mean:

$$
\operatorname{skewness}
\sim
\frac{1}{n}\sum_{i=1}^{n}
\left(\frac{x_i-\bar{x}}{s}\right)^3
$$

Because the differences are raised to the third power, positive and negative differences do not cancel each other out. Large differences also receive considerably more weight.

The `pandas.Series.skew()` method uses an adjusted calculation that takes the size of the dataset into account.

### Activity 2 — Read the statistical summary

Answer briefly:

1. What price would you report as the **typical** sale price: the mean or the median? Why?
2. Between which two values does the middle 50% of sale prices lie?
3. What does the positive skewness suggest about the distribution?


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].hist(sale_price / 1_000, bins=30, edgecolor="black")
axes[0].axvline(sale_price.mean() / 1_000, linestyle="--", label="Mean")
axes[0].axvline(median / 1_000, linestyle="-", label="Median")
axes[0].axvline(q1 / 1_000, linestyle=":", label="Q1 and Q3")
axes[0].axvline(q3 / 1_000, linestyle=":")
axes[0].set(
    title="Distribution of sale prices",
    xlabel="Sale price [thousand USD]",
    ylabel="Number of houses",
)
axes[0].legend()

sns.boxplot(x=sale_price / 1_000, ax=axes[1])
axes[1].set(
    title="Sale-price boxplot",
    xlabel="Sale price [thousand USD]",
)

plt.tight_layout()
plt.show()


## READING: Histogram and Box Plot

### Histogram

A **histogram** shows how numerical values are distributed. It divides the values into intervals called **bins** and displays how many observations fall into each interval.

A histogram helps us see:

- where most values are concentrated,
- whether the distribution is symmetrical or skewed,
- whether the data contain one or more peaks,
- whether unusually low or high values may be present.

The appearance of a histogram depends on the selected number and width of bins.

### Box Plot

A **box plot** provides a compact summary of a numerical variable using its median and quartiles.

It displays:

- the **median** as a line inside the box,
- the first quartile ($Q_1$) and third quartile ($Q_3$) as the boundaries of the box,
- the middle 50% of values inside the box,
- **whiskers**, which usually extend to the most extreme values within $1.5 \cdot IQR$ from the box,
- potential **outliers** as individual points beyond the whiskers.

A box plot is useful for identifying potential outliers and comparing the distributions of several groups. Unlike a histogram, however, it does not show the detailed shape of the distribution.

### Interpretation checkpoint

Write three short sentences:

1. describe the shape of the distribution,
2. explain the difference between the mean and median,
3. state one thing visible in the histogram but not clearly visible in the boxplot.


### Verification challenge — one extreme value

Before running the cell, predict which quantities will change most when one sale price is replaced by 2,000,000 USD.


In [ ]:
counterfactual_price = sale_price.copy()
counterfactual_price.loc[counterfactual_price.idxmax()] = 2_000_000

comparison = pd.DataFrame({
    "original": {
        "mean": sale_price.mean(),
        "median": sale_price.median(),
        "standard_deviation": sale_price.std(),
        "IQR": sale_price.quantile(0.75) - sale_price.quantile(0.25),
    },
    "after one extreme value": {
        "mean": counterfactual_price.mean(),
        "median": counterfactual_price.median(),
        "standard_deviation": counterfactual_price.std(),
        "IQR": (
            counterfactual_price.quantile(0.75)
            - counterfactual_price.quantile(0.25)
        ),
    },
})

comparison["absolute_change"] = (
    comparison["after one extreme value"] - comparison["original"]
)

comparison


Compare the result with your prediction.

- Which measures changed most?
- Which measures were almost unchanged?
- Does this experiment prove that the median is always better than the mean? Explain.


## 3. Numbers can represent ordered categories

`OverallQual` and `OverallCond` are stored as integers from 1 to 10. They have an order, but the distance between neighbouring ratings is not guaranteed to be equal.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x="OverallQual", y="SalePrice", ax=axes[0])
axes[0].set(
    title="Sale price across quality ratings",
    xlabel="Overall quality rating",
    ylabel="Sale price [USD]",
)

sns.boxplot(data=df, x="OverallCond", y="SalePrice", ax=axes[1])
axes[1].set(
    title="Sale price across condition ratings",
    xlabel="Overall condition rating",
    ylabel="Sale price [USD]",
)

plt.tight_layout()
plt.show()


### Activity 3 — Compare quality and condition

1. Which variable shows clearer separation of sale prices across its levels?
2. Is the relationship perfectly increasing at every step?
3. Why is a boxplot more informative here than a line joining the group means?
4. Complete the sentence carefully:

> Houses with a higher `OverallQual` rating tend to ...


In [ ]:
quality_summary = (
    df.groupby("OverallQual", observed=True)
      .agg(
          houses=("Id", "count"),
          median_price=("SalePrice", "median"),
          mean_price=("SalePrice", "mean"),
      )
)

quality_summary


## 4. Missing values can have different meanings

A missing value may mean:

- the value is unknown,
- the value was not recorded,
- the variable is not applicable,
- the real-world feature is absent.

We will investigate `GarageFinish` instead of replacing its missing values immediately.


In [ ]:
garage_columns = [
    "GarageType",
    "GarageFinish",
    "GarageYrBlt",
    "GarageCars",
    "GarageArea",
]

garage = df_full.loc[:, garage_columns].copy()
garage["GarageFinishMissing"] = garage["GarageFinish"].isna()

print("Missing values in garage-related columns:")
display(garage.isna().sum().to_frame("missing_count"))

print("Numerical garage properties by GarageFinish missingness:")
display(
    garage.groupby("GarageFinishMissing")[["GarageCars", "GarageArea"]]
          .agg(["count", "median", "min", "max"])
)

print("GarageFinish × GarageType:")
display(
    pd.crosstab(
        garage["GarageFinish"].fillna("<missing>"),
        garage["GarageType"].fillna("<missing>"),
        dropna=False,
    )
)


### Activity 4 — Build an evidence chain

Complete the statements:

1. **Initial hypothesis:** Missing `GarageFinish` may mean ...
2. **Evidence from `GarageArea` and `GarageCars`:** ...
3. **Evidence from `GarageType`:** ...
4. **Most defensible interpretation:** ...

In [ ]:
missing_finish_with_positive_area = garage.loc[
    garage["GarageFinish"].isna()
    & garage["GarageArea"].fillna(0).gt(0)
]

print(
    "Rows with missing GarageFinish but positive GarageArea:",
    len(missing_finish_with_positive_area),
)

garage["GarageFinishEDA"] = garage["GarageFinish"].fillna("NoGarage")
garage["GarageFinishEDA"].value_counts(dropna=False)


### Stop and explain

Why is `NoGarage` a defensible category for this particular variable? Why would the rule “replace every missing categorical value with `No...`” be unsafe?


## 5. Choose a graph for the question

A graph should be selected from the analytical question, not from habit.


### Plot-selection activity

Match each question with a suitable first visualisation.

| Question | Choose from |
|---|---|
| A. What is the distribution of one numerical variable? | histogram / boxplot / scatter plot / count plot |
| B. How does a numerical variable differ across categories? | histogram / boxplot / scatter plot / count plot |
| C. How are two numerical variables related? | histogram / boxplot / scatter plot / count plot |
| D. How many observations belong to each category? | histogram / boxplot / scatter plot / count plot |

Then choose one question by changing `QUESTION_ID` in the next cell.


In [ ]:
QUESTION_ID = "D"  # Choose "A", "B", "C", or "D".

fig, ax = plt.subplots(figsize=(9, 5))

if QUESTION_ID == "A":
    sns.histplot(data=df, x="SalePrice", bins=30, ax=ax)
    ax.set(title="Distribution of sale price", xlabel="Sale price [USD]")

elif QUESTION_ID == "B":
    category_order = (
        df.groupby("BldgType")["SalePrice"]
          .median()
          .sort_values()
          .index
    )
    sns.boxplot(
        data=df,
        x="BldgType",
        y="SalePrice",
        order=category_order,
        ax=ax,
    )
    ax.set(
        title="Sale price across building types",
        xlabel="Building type",
        ylabel="Sale price [USD]",
    )

elif QUESTION_ID == "C":
    sns.scatterplot(
        data=df,
        x="GrLivArea",
        y="SalePrice",
        alpha=0.65,
        ax=ax,
    )
    ax.set(
        title="Living area and sale price",
        xlabel="Above-ground living area [ft²]",
        ylabel="Sale price [USD]",
    )

elif QUESTION_ID == "D":
    count_order = df["BldgType"].value_counts().index
    sns.countplot(data=df, x="BldgType", order=count_order, ax=ax)
    ax.set(
        title="Number of houses by building type",
        xlabel="Building type",
        ylabel="Number of houses",
    )

else:
    raise ValueError("QUESTION_ID must be A, B, C, or D.")

plt.tight_layout()
plt.show()


### Activity 5 — Read the plot as evidence

For your selected plot, write:

1. the analytical question,
2. the main visible pattern,
3. one quantity or relationship that the plot does **not** show,
4. one table or second plot that could verify your interpretation.


In [ ]:
building_type_summary = (
    df.groupby("BldgType", observed=True)
      .agg(
          houses=("Id", "count"),
          median_price=("SalePrice", "median"),
          median_area=("GrLivArea", "median"),
          price_IQR=(
              "SalePrice",
              lambda values: values.quantile(0.75) - values.quantile(0.25),
          ),
      )
      .sort_values("median_price", ascending=False)
)

building_type_summary


## 6. Correlation is a summary, not a conclusion

Correlation describes association. It does not establish causality, and it may hide nonlinear patterns, groups, or unusual observations.

## READING: Pearson and Spearman Correlation

### Pearson Correlation

The **Pearson correlation coefficient** measures the strength and direction of a **linear relationship** between two numerical variables.

It can be calculated as:

$$
r =
\frac{
\sum_{i=1}^{n}(x_i-\bar{x})(y_i-\bar{y})
}{
\sqrt{\sum_{i=1}^{n}(x_i-\bar{x})^2}
\sqrt{\sum_{i=1}^{n}(y_i-\bar{y})^2}
}
$$

The coefficient ranges from $-1$ to $1$:

- $r = 1$: a perfect positive linear relationship,
- $r = 0$: no linear relationship,
- $r = -1$: a perfect negative linear relationship.

Pearson correlation can also be understood as the cosine of the angle between two mean-centred data vectors. It is sensitive to outliers and may fail to describe strong relationships that are not linear.

---

### Spearman Correlation

The **Spearman correlation coefficient** measures the strength and direction of a **monotonic relationship** between two variables. A monotonic relationship means that as one variable increases, the other variable tends to consistently increase or consistently decrease, although not necessarily at a constant rate.

Spearman correlation is calculated by replacing the original values with their ranks and then calculating Pearson correlation between the ranks:

$$
r_s = \operatorname{Pearson}(R_x,R_y)
$$

where $R_x$ and $R_y$ are the ranks of the values in variables $x$ and $y$.

The coefficient also ranges from $-1$ to $1$:

- $r_s = 1$: the rankings are exactly the same,
- $r_s = 0$: no clear monotonic relationship,
- $r_s = -1$: the rankings are exactly reversed.

Spearman correlation is useful for ordinal data, non-linear but monotonic relationships, and data containing outliers.

---

### Pearson or Spearman?

- Use **Pearson correlation** when you want to measure a linear relationship between numerical variables.
- Use **Spearman correlation** when the relationship may be non-linear but monotonic, when the data are ordinal, or when outliers strongly affect Pearson correlation.

Correlation describes an association between variables, but it does not prove that one variable causes changes in the other.

### Predict before running

1. Which selected variable do you expect to have the strongest positive association with `SalePrice`?
2. Which variable is ordinal rather than continuous?
3. Why should `MSSubClass` not be added automatically just because it is stored as an integer?


In [ ]:
correlation_features = [
    "LotArea",
    "YearBuilt",
    "GrLivArea",
    "GarageArea",
    "OverallQual",
    "OverallCond",
    "SalePrice",
]

pearson_corr = df_full[correlation_features].corr(method="pearson")
spearman_corr = df_full[correlation_features].corr(method="spearman")

sale_price_correlations = pd.DataFrame({
    "Pearson": pearson_corr["SalePrice"],
    "Spearman": spearman_corr["SalePrice"],
}).drop(index="SalePrice")

sale_price_correlations["absolute_difference"] = (
    sale_price_correlations["Pearson"]
    - sale_price_correlations["Spearman"]
).abs()

sale_price_correlations.sort_values("Spearman", ascending=False)


In [ ]:
plt.figure(figsize=(9, 7))

sns.heatmap(
    pearson_corr,
    annot=True,
    fmt=".2f",
    vmin=-1,
    vmax=1,
    center=0,
    square=True,
)

plt.title("Pearson correlation matrix")
plt.tight_layout()
plt.show()


### Activity 6 — Interpret and verify correlation

1. Which variable has the strongest Pearson association with `SalePrice`?
2. Is the ordering exactly the same for Spearman correlation?
3. Choose one variable for which Pearson and Spearman differ noticeably. What property of the relationship might explain the difference?
4. Rewrite this statement so that it is defensible:

> `OverallQual` has the largest correlation, so improving quality by one point causes the sale price to rise by a fixed amount.


In [ ]:
VARIABLE_TO_VERIFY = "GrLivArea"  # Try "OverallQual" or another suitable variable.

fig, ax = plt.subplots(figsize=(8, 5))

if VARIABLE_TO_VERIFY in {"OverallQual", "OverallCond"}:
    sns.boxplot(
        data=df_full,
        x=VARIABLE_TO_VERIFY,
        y="SalePrice",
        ax=ax,
    )
else:
    sns.scatterplot(
        data=df_full,
        x=VARIABLE_TO_VERIFY,
        y="SalePrice",
        alpha=0.65,
        ax=ax,
    )

ax.set(
    title=f"Verification plot: {VARIABLE_TO_VERIFY} and SalePrice",
    ylabel="Sale price [USD]",
)
plt.tight_layout()
plt.show()


### Stop and explain

What does the verification plot reveal that the single correlation coefficient does not?


## 7. Outliers are investigation targets

The IQR rule can flag unusual values. It cannot decide whether a value is incorrect.

Possible outcomes include:

- keep the observation,
- correct a documented error,
- analyse the data with and without it,
- remove it only with a clear domain-based reason.


In [ ]:
def iqr_outlier_flag(series, multiplier=1.5):
    clean = series.dropna()
    q1 = clean.quantile(0.25)
    q3 = clean.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - multiplier * iqr
    upper = q3 + multiplier * iqr

    flag = series.lt(lower) | series.gt(upper)

    return flag, {
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "lower_bound": lower,
        "upper_bound": upper,
    }


sale_flag, sale_bounds = iqr_outlier_flag(df_full["SalePrice"])
area_flag, area_bounds = iqr_outlier_flag(df_full["GrLivArea"])

outlier_work = df_full.copy()
outlier_work["SalePriceOutlier"] = sale_flag
outlier_work["GrLivAreaOutlier"] = area_flag
outlier_work["AnyIQRFlag"] = sale_flag | area_flag

pd.DataFrame({
    "SalePrice": sale_bounds,
    "GrLivArea": area_bounds,
})


In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

sns.scatterplot(
    data=outlier_work,
    x="GrLivArea",
    y="SalePrice",
    hue="AnyIQRFlag",
    alpha=0.75,
    ax=ax,
)

ax.set(
    title="IQR-flagged observations",
    xlabel="Above-ground living area [ft²]",
    ylabel="Sale price [USD]",
)
plt.tight_layout()
plt.show()

inspection_columns = [
    "Id",
    "Neighborhood",
    "BldgType",
    "OverallQual",
    "OverallCond",
    "YearBuilt",
    "GrLivArea",
    "GarageArea",
    "SaleCondition",
    "SalePrice",
    "SalePriceOutlier",
    "GrLivAreaOutlier",
]

flagged_examples = (
    outlier_work.loc[outlier_work["AnyIQRFlag"], inspection_columns]
                .sort_values(["GrLivArea", "SalePrice"], ascending=False)
)

flagged_examples.head(15)


### Activity 7 — Investigate one unusual observation

Choose one row from `flagged_examples` and answer:

1. Why was it flagged?
2. Are its values internally plausible?
3. Which related variables help explain it?
4. Could it strongly influence a correlation or later model?
5. What is your current decision: keep, correct, remove, or analyse both ways?

A valid decision may be: **keep it for now and perform a sensitivity analysis later**.


## 8. Your own investigation

Choose **one** question:

- Is house quality more strongly associated with price than house condition?
- Do larger houses always sell for more?
- Which building types show the greatest variation in sale price?
- Are newer houses generally rated as higher quality?
- Does a missing garage description usually indicate that the house has no garage?

Your investigation must include:

1. a clear analytical question,
2. an expectation written before analysis,
3. one suitable numerical summary,
4. one visualisation,
5. a short interpretation,
6. one verification step,
7. one limitation or alternative explanation.


In [ ]:
# Investigation title:
# Question:
# Expected result before analysis:
# Variables and their analytical roles:
# Why the selected summary and plot fit the question:

# Reuse and adapt one of the earlier analysis patterns here.


## Investigation summary

**Question:**  

**What I expected:**  

**What I found:**  

**How I verified it:**  

**One limitation or alternative explanation:**


### Optional AI reviewer — only after your own draft

After writing the summary above, you may use this prompt:

```text
Act as a careful statistics tutor.
Do not rewrite my answer and do not provide a model answer.
Read my short interpretation and ask me exactly two questions:
1. one question about whether my evidence supports my claim,
2. one question about a limitation or alternative explanation.
Wait for my answer after each question.
```

Record only:

**One claim I kept unchanged and why:**  

**One claim I revised and why:**  

Do not paste the whole conversation.


# End-of-lab self-check

Before finishing, check that you can:

- [ ] distinguish `dtype` from analytical role,
- [ ] explain when the median and IQR are useful,
- [ ] describe a right-skewed distribution,
- [ ] interpret an ordinal rating cautiously,
- [ ] investigate the meaning of a missing value before filling it,
- [ ] select a plot based on a question,
- [ ] explain why correlation does not prove causality,
- [ ] use a second output to verify a claim,
- [ ] investigate an outlier before deciding what to do with it,
- [ ] explain what you learned from AI without reading the chat transcript.

These skills form the EDA part of the semester project. Exercise 3 will continue with scaling, distance, and K-means clustering.
